# CatBoost + LightGBM Ensemble

This notebook tests whether combining the two strongest standalone
models improves ROC-AUC.

Current results:

- CatBoost OOF: **0.941711**
- CatBoost Public LB: **0.941560**
- LightGBM OOF: **0.941770**
- LightGBM Public LB: **0.941650**

Ensembling will be evaluated using saved out-of-fold predictions
before creating any new Kaggle submission.

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score

In [2]:
cat_oof = pd.read_csv(
    "../outputs/predictions/catboost_oof.csv"
)

lgb_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_oof.csv"
)

cat_test = pd.read_csv(
    "../outputs/predictions/catboost_test.csv"
)

lgb_test = pd.read_csv(
    "../outputs/predictions/lightgbm_test.csv"
)

In [3]:
cat_oof.shape, lgb_oof.shape, cat_test.shape, lgb_test.shape

((668665, 3), (668665, 3), (286571, 2), (286571, 2))

In [4]:
assert cat_oof["id"].equals(lgb_oof["id"])
assert cat_oof["target"].equals(lgb_oof["target"])
assert cat_test["id"].equals(lgb_test["id"])

print("Prediction files aligned correctly.")

Prediction files aligned correctly.


In [5]:
y = cat_oof["target"]

cat_pred = cat_oof["catboost_oof"]
lgb_pred = lgb_oof["lightgbm_oof"]

cat_auc = roc_auc_score(y, cat_pred)
lgb_auc = roc_auc_score(y, lgb_pred)

print(f"CatBoost OOF: {cat_auc:.6f}")
print(f"LightGBM OOF: {lgb_auc:.6f}")

CatBoost OOF: 0.941711
LightGBM OOF: 0.941770


In [6]:
prediction_correlation = np.corrcoef(
    cat_pred,
    lgb_pred
)[0, 1]

prediction_correlation

np.float64(0.9974373107398214)

In [7]:
blend_50 = (
    0.5 * cat_pred
    + 0.5 * lgb_pred
)

blend_50_auc = roc_auc_score(
    y,
    blend_50
)

blend_50_auc

0.9419061547370386

In [8]:
blend_results = []

for w in np.arange(0.0, 1.01, 0.05):

    blended = (
        w * lgb_pred
        + (1 - w) * cat_pred
    )

    auc = roc_auc_score(
        y,
        blended
    )

    blend_results.append({
        "LightGBM_Weight": w,
        "CatBoost_Weight": 1 - w,
        "OOF_AUC": auc
    })

blend_results = pd.DataFrame(
    blend_results
)

blend_results.sort_values(
    "OOF_AUC",
    ascending=False
).head(10)

,LightGBM_Weight,CatBoost_Weight,OOF_AUC
11,0.55,0.45,0.941907
10,0.50,0.50,0.941906
12,0.60,0.40,0.941906
9,0.45,0.55,0.941902
13,0.65,0.35,0.941900
8,0.40,0.60,0.941894
14,0.70,0.30,0.941892
7,0.35,0.65,0.941882
15,0.75,0.25,0.941880
6,0.30,0.70,0.941868


In [9]:
best_probability_blend = (
    blend_results
    .sort_values(
        "OOF_AUC",
        ascending=False
    )
    .iloc[0]
)

best_probability_blend

LightGBM_Weight    0.550000
CatBoost_Weight    0.450000
OOF_AUC            0.941907
Name: 11, dtype: float64

In [10]:
cat_rank = cat_pred.rank(
    method="average",
    pct=True
)

lgb_rank = lgb_pred.rank(
    method="average",
    pct=True
)

In [11]:
rank_results = []

for w in np.arange(0.0, 1.01, 0.05):

    blended_rank = (
        w * lgb_rank
        + (1 - w) * cat_rank
    )

    auc = roc_auc_score(
        y,
        blended_rank
    )

    rank_results.append({
        "LightGBM_Weight": w,
        "CatBoost_Weight": 1 - w,
        "OOF_AUC": auc
    })

rank_results = pd.DataFrame(
    rank_results
)

rank_results.sort_values(
    "OOF_AUC",
    ascending=False
).head(10)

,LightGBM_Weight,CatBoost_Weight,OOF_AUC
11,0.55,0.45,0.941911
10,0.50,0.50,0.941910
12,0.60,0.40,0.941909
9,0.45,0.55,0.941906
13,0.65,0.35,0.941904
8,0.40,0.60,0.941898
14,0.70,0.30,0.941895
7,0.35,0.65,0.941886
15,0.75,0.25,0.941882
6,0.30,0.70,0.941871


In [12]:
best_rank_blend = (
    rank_results
    .sort_values(
        "OOF_AUC",
        ascending=False
    )
    .iloc[0]
)

best_rank_blend

LightGBM_Weight    0.550000
CatBoost_Weight    0.450000
OOF_AUC            0.941911
Name: 11, dtype: float64

In [13]:
results_summary = pd.DataFrame({
    "Method": [
        "CatBoost",
        "LightGBM",
        "50/50 Probability Blend",
        "Best Probability Blend",
        "Best Rank Blend"
    ],
    "OOF_AUC": [
        cat_auc,
        lgb_auc,
        blend_50_auc,
        best_probability_blend["OOF_AUC"],
        best_rank_blend["OOF_AUC"]
    ]
})

results_summary.sort_values(
    "OOF_AUC",
    ascending=False
)

,Method,OOF_AUC
4,Best Rank Blend,0.941911
3,Best Probability Blend,0.941907
2,50/50 Probability Blend,0.941906
1,LightGBM,0.941770
0,CatBoost,0.941711


In [14]:
cat_test_pred = cat_test["catboost_prediction"]
lgb_test_pred = lgb_test["lightgbm_prediction"]

In [15]:
cat_test_rank = cat_test_pred.rank(
    method="average",
    pct=True
)

lgb_test_rank = lgb_test_pred.rank(
    method="average",
    pct=True
)

In [16]:
ensemble_test_predictions = (
    0.55 * lgb_test_rank
    + 0.45 * cat_test_rank
)

In [17]:
ensemble_test_predictions.head(10)

0    0.424112
1    0.503381
2    0.320747
3    0.274989
4    0.511146
5    0.306002
6    0.790004
7    0.065871
8    0.887577
9    0.327762
dtype: float64

In [18]:
ensemble_test_predictions.min(), ensemble_test_predictions.max()

(np.float64(0.00012073796720533482), np.float64(0.9999961615097133))

In [19]:
submission_ensemble = pd.DataFrame({
    "id": cat_test["id"],
    "Will_Buy_EV": ensemble_test_predictions
})

In [20]:
submission_ensemble.head()

,id,Will_Buy_EV
0,668665,0.424112
1,668666,0.503381
2,668667,0.320747
3,668668,0.274989
4,668669,0.511146


In [21]:
submission_ensemble.shape

(286571, 2)

In [22]:
submission_ensemble.isna().sum()

id             0
Will_Buy_EV    0
dtype: int64

In [23]:
submission_ensemble["id"].equals(cat_test["id"])

True

In [24]:
submission_ensemble.to_csv(
    "../outputs/submissions/submission_003_cat_lgb_rank_blend.csv",
    index=False
)

## Results

CatBoost and LightGBM OOF predictions had a correlation of
**0.997437**, indicating that the models produce very similar
predictions.

Despite this, blending improved ROC-AUC.

### OOF Results

- CatBoost: **0.941711**
- LightGBM: **0.941770**
- 50/50 probability blend: **0.941906**
- 55/45 probability blend: **0.941907**
- 55/45 rank blend: **0.941911**

The best local ensemble used:

- **55% LightGBM**
- **45% CatBoost**
- percentile-rank predictions

### Kaggle Result

Public leaderboard ROC-AUC:

**0.941690**

This improved the previous best public score of **0.941650**.

The ensemble therefore improved both local validation and the
public leaderboard, although the leaderboard improvement was
smaller than the local OOF improvement.